# Анализ доходности и волатильности криптовалют

**Цель:** сравнить доходность, риск и взаимосвязи BTC, ETH, BNB, SOL, XRP, DOGE, ADA и TRX за 365 дней по данным CoinGecko API.

**Гипотезы:**
1. BTC и ETH менее волатильны большинства альткоинов.
2. Более высокая доходность связана с более высоким риском.
3. Доходности криптовалют положительно коррелируют.
4. DOGE нестабильнее крупных инфраструктурных криптовалют.


In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from dotenv import load_dotenv

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})


## 1. Сбор данных

Используется CoinGecko Demo API. Ключ хранится в переменной окружения `COINGECKO_API_KEY`. Без ключа ноутбук читает `raw_crypto_data.csv`.


In [ ]:
load_dotenv()

API_KEY = os.getenv("COINGECKO_API_KEY")
DATA_FILE = "raw_crypto_data.csv"
USE_LOCAL_DATA = API_KEY is None
BASE_URL = "https://api.coingecko.com/api/v3"
headers = {"x-cg-demo-api-key": API_KEY} if API_KEY else {}

if USE_LOCAL_DATA and not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        "Добавьте COINGECKO_API_KEY или файл raw_crypto_data.csv рядом с ноутбуком."
    )

coins = {
    "BTC": "bitcoin", "ETH": "ethereum", "BNB": "binancecoin",
    "SOL": "solana", "XRP": "ripple", "DOGE": "dogecoin",
    "ADA": "cardano", "TRX": "tron",
}


In [ ]:
def get_coin_market_data(coin_id: str, symbol: str, days: int = 365) -> pd.DataFrame | None:
    """Загружает цену, капитализацию и объём торгов одной криптовалюты."""
    url = f"{BASE_URL}/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": days, "interval": "daily"}
    response = requests.get(url, params=params, headers=headers, timeout=30)

    if response.status_code != 200:
        print(f"Ошибка для {symbol}: {response.status_code}")
        return None

    data = response.json()
    prices = pd.DataFrame(data["prices"], columns=["timestamp", "price"])
    caps = pd.DataFrame(data["market_caps"], columns=["timestamp", "market_cap"])
    volumes = pd.DataFrame(data["total_volumes"], columns=["timestamp", "total_volume"])

    frame = prices.merge(caps, on="timestamp").merge(volumes, on="timestamp")
    frame["date"] = pd.to_datetime(frame["timestamp"], unit="ms").dt.date
    frame["symbol"] = symbol
    frame["coin_id"] = coin_id
    return frame[["date", "symbol", "coin_id", "price", "market_cap", "total_volume"]]


if USE_LOCAL_DATA:
    crypto_data = pd.read_csv(DATA_FILE)
else:
    frames = []
    for symbol, coin_id in coins.items():
        print(f"Загружаю {symbol}...")
        frame = get_coin_market_data(coin_id, symbol)
        if frame is not None:
            frames.append(frame)
        time.sleep(1.2)

    if not frames:
        raise RuntimeError("CoinGecko API не вернул данные.")
    crypto_data = pd.concat(frames, ignore_index=True)
    crypto_data.to_csv(DATA_FILE, index=False)

print("Размер исходного датасета:", crypto_data.shape)
crypto_data.head()


## 2. Первичный осмотр и очистка

Проверяются структура, типы, пропуски и дубликаты. Повторные комбинации `symbol + date` удаляются.


In [ ]:
display(crypto_data.head())
crypto_data.info()
print("Пропуски:", crypto_data.isna().sum(), sep="\n")
print("Полные дубликаты:", crypto_data.duplicated().sum())


In [ ]:
crypto_clean = crypto_data.copy()
crypto_clean["date"] = pd.to_datetime(crypto_clean["date"])

numeric_columns = ["price", "market_cap", "total_volume"]
for column in numeric_columns:
    crypto_clean[column] = pd.to_numeric(crypto_clean[column], errors="coerce")

crypto_clean = (
    crypto_clean
    .sort_values(["symbol", "date"])
    .drop_duplicates(subset=["symbol", "date"], keep="last")
    .reset_index(drop=True)
)

crypto_clean[numeric_columns] = (
    crypto_clean.groupby("symbol")[numeric_columns]
    .transform(lambda values: values.ffill().bfill())
)
crypto_clean = crypto_clean.dropna(subset=numeric_columns)

quality = crypto_clean.groupby("symbol").agg(
    start_date=("date", "min"),
    end_date=("date", "max"),
    observations=("date", "count"),
    min_price=("price", "min"),
    max_price=("price", "max"),
)
display(quality)


## 3. Создание признаков

Рассчитываются дневная и накопленная доходность, а также 30-дневная волатильность, приведённая к годовому масштабу.


In [ ]:
crypto_clean["daily_return"] = (
    crypto_clean.groupby("symbol")["price"].pct_change()
)
crypto_clean["cumulative_return"] = (
    crypto_clean.groupby("symbol")["price"]
    .transform(lambda prices: prices / prices.iloc[0] - 1)
)
crypto_clean["volatility_30d"] = (
    crypto_clean.groupby("symbol")["daily_return"]
    .transform(lambda returns: returns.rolling(30).std() * np.sqrt(365))
)
crypto_clean["daily_return_pct"] = crypto_clean["daily_return"] * 100
crypto_clean["cumulative_return_pct"] = crypto_clean["cumulative_return"] * 100
crypto_clean["volatility_30d_pct"] = crypto_clean["volatility_30d"] * 100


## 4. Сводная аналитическая таблица


In [ ]:
def calculate_max_drawdown(prices: pd.Series) -> float:
    """Возвращает максимальное падение цены от предыдущего максимума."""
    drawdown = prices / prices.cummax() - 1
    return drawdown.min()


price_summary = crypto_clean.groupby("symbol").agg(
    start_date=("date", "min"),
    end_date=("date", "max"),
    start_price=("price", "first"),
    end_price=("price", "last"),
    avg_market_cap=("market_cap", "mean"),
    avg_total_volume=("total_volume", "mean"),
).reset_index()

return_summary = (
    crypto_clean.dropna(subset=["daily_return"])
    .groupby("symbol")
    .agg(
        mean_daily_return=("daily_return", "mean"),
        median_daily_return=("daily_return", "median"),
        daily_volatility=("daily_return", "std"),
    )
    .reset_index()
)

drawdowns = (
    crypto_clean.groupby("symbol")["price"]
    .apply(calculate_max_drawdown)
    .reset_index(name="max_drawdown")
)

summary = price_summary.merge(return_summary, on="symbol").merge(drawdowns, on="symbol")
summary["cumulative_return"] = summary["end_price"] / summary["start_price"] - 1
summary["annualized_volatility"] = summary["daily_volatility"] * np.sqrt(365)
summary["return_to_risk"] = summary["cumulative_return"] / summary["annualized_volatility"]

summary_display = summary.copy()
for column in [
    "mean_daily_return", "median_daily_return", "daily_volatility",
    "max_drawdown", "cumulative_return", "annualized_volatility",
]:
    summary_display[column] *= 100

summary_display = summary_display.sort_values("cumulative_return", ascending=False).round(3)
display(summary_display)


## 5. Визуализация


In [ ]:
price_pivot = crypto_clean.pivot(index="date", columns="symbol", values="price")
normalized_prices = price_pivot / price_pivot.iloc[0] * 100

plt.figure(figsize=(12, 6))
for symbol in normalized_prices.columns:
    plt.plot(normalized_prices.index, normalized_prices[symbol], label=symbol)
plt.title("Динамика нормализованных цен криптовалют")
plt.xlabel("Дата")
plt.ylabel("Индекс цены, первый день = 100")
plt.legend(ncol=4)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
chart_data = summary.sort_values("cumulative_return")
plt.figure(figsize=(10, 6))
plt.barh(chart_data["symbol"], chart_data["cumulative_return"] * 100)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Накопленная доходность криптовалют за период")
plt.xlabel("Накопленная доходность, %")
plt.ylabel("Криптовалюта")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
chart_data = summary.sort_values("annualized_volatility")
plt.figure(figsize=(10, 6))
plt.barh(chart_data["symbol"], chart_data["annualized_volatility"] * 100)
plt.title("Годовая волатильность криптовалют")
plt.xlabel("Годовая волатильность, %")
plt.ylabel("Криптовалюта")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
rolling = crypto_clean.pivot(index="date", columns="symbol", values="volatility_30d_pct")
plt.figure(figsize=(12, 6))
for symbol in rolling.columns:
    plt.plot(rolling.index, rolling[symbol], label=symbol)
plt.title("30-дневная скользящая волатильность")
plt.xlabel("Дата")
plt.ylabel("Годовая волатильность, %")
plt.legend(ncol=4)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
x = summary["annualized_volatility"] * 100
y = summary["cumulative_return"] * 100
plt.scatter(x, y)
for _, row in summary.iterrows():
    plt.annotate(
        row["symbol"],
        (row["annualized_volatility"] * 100, row["cumulative_return"] * 100),
        xytext=(5, 5),
        textcoords="offset points",
    )
plt.title("Соотношение риска и доходности")
plt.xlabel("Годовая волатильность, %")
plt.ylabel("Накопленная доходность, %")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
returns_pivot = crypto_clean.pivot(index="date", columns="symbol", values="daily_return")
correlation_matrix = returns_pivot.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)

plt.figure(figsize=(9, 7))
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
plt.title("Корреляционная матрица дневных доходностей")
plt.xlabel("Криптовалюта")
plt.ylabel("Криптовалюта")
plt.tight_layout()
plt.show()


## 6. Проверка гипотез и выводы

1. **BTC и ETH менее волатильны:** частично подтверждено. BTC был стабильнее большинства альткоинов, но ETH — нет.
2. **Больший риск связан с большей доходностью:** не подтверждено. TRX дал лучший результат при минимальной волатильности, а ADA и DOGE были рискованными и убыточными.
3. **Доходности положительно коррелируют:** подтверждено на выбранной выборке; TRX связан с рынком слабее остальных.
4. **DOGE нестабилен:** подтверждено; DOGE показал максимальную волатильность.

**Ограничения:** один год наблюдений; исторический анализ не является прогнозом; `return_to_risk` не заменяет коэффициент Шарпа; последний день может быть неполным; результаты не являются инвестиционной рекомендацией.
